In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import time

# Import Sionna RT components
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, CoverageMap

# For link-level simulations
from sionna.channel import cir_to_ofdm_channel, cir_to_time_channel,subcarrier_frequencies, OFDMChannel, ApplyOFDMChannel, CIRDataset
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.utils import compute_ber, ebnodb2no, PlotBER
from sionna.ofdm import KBestDetector, LinearDetector
from sionna.mimo import StreamManagement

#matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import time
import sionna
# Import Sionna RT components


# For link-level simulations
from sionna.channel import OFDMChannel

from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.utils import compute_ber, ebnodb2no, PlotBER
from sionna.ofdm import KBestDetector, LinearDetector
from sionna.mimo import StreamManagement



In [12]:
import matplotlib.pyplot as plt
import numpy as np
import time

# Import Sionna RT components

# For link-level simulationst

#matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import time
import sionna
import sionna.rt
# Import Sionna RT components
from sionna.rt import load_scene

ImportError: cannot import name 'load_scene' from 'sionna.rt' (unknown location)

In [8]:
scene = load_scene() # Try also sionna.rt.scene.etoile

scene.frequency = 3.5e9 # in Hz; implicitly updates RadioMaterials

scene.synthetic_array = False # If set to False, ray tracing will be done per antenna element (slower for large arrays)

NameError: name 'load_scene' is not defined

In [ ]:
scene.remove("tx")
scene.remove("rx")
scene.remove("tx1")
# Configure antenna array for all transmitters
scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=2.0,
                             horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="V")

# Configure antenna array for all receivers
scene.rx_array = PlanarArray(num_rows=4,
                             num_cols=4,
                             vertical_spacing=2.0,
                             horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="cross")

# Create transmitter


tx = Transmitter(name="tx",
                 position=[100,120,30])
# Create a receiver
scene.add(tx)

rx = Receiver(name="rx",
                 position=[50,50,30])

# Add transmitter instance to scene
scene.add(rx)


 # Transmitter points towards receiver

In [ ]:
scene.remove("tx1")

tx1 = Transmitter(name="tx1",
                 position=[30,21,27])
# Add transmitter instance to scene
scene.add(tx1)

In [ ]:
scene.preview()

In [ ]:
paths = scene.compute_paths(max_depth=3) 
# 取得 Tx 到 Rx 的 CIR
subcarrier_spacing = 30e3
fft_size = 408
print("Shape of `a` before applying Doppler shifts: ", paths.a.shape)

# Apply Doppler shifts
paths.apply_doppler(sampling_frequency=subcarrier_spacing,num_time_steps=10, tx_velocities=[3.,0,0],rx_velocities=[0,7.,0]) 
print("Shape of `a` after applying Doppler shifts: ", paths.a.shape)

a, tau = paths.cir()
print("Shape of tau: ", tau.shape)

frequencies = subcarrier_frequencies(fft_size, subcarrier_spacing)


h_freq= cir_to_ofdm_channel(frequencies,
                             a,
                             tau,
                             normalize=False) # Non-normalized includes path-loss

# Verify that the channel power is normalized


print("Shape of h_freq: ", h_freq.shape)

In [ ]:
# OFDM system parameters
num_subcarriers = 1024
subcarrier_spacing=30e3

# Compute frequencies of subcarriers relative to the carrier frequency
frequencies = subcarrier_frequencies(num_subcarriers, subcarrier_spacing)

# Compute channel frequency response
h_freq = paths.cfr(frequencies=frequencies,
                   normalize=True, # Normalize energy
                   normalize_delays=True,
                   out_type="numpy")

# Shape: [num_rx, num_rx_ant, num_tx, num_tx_ant, num_time_steps, num_subcarriers]
print("Shape of h_freq: ", h_freq.shape)

# Plot absolute value
plt.figure()
plt.plot(np.abs(h_freq)[0,0,0,0,0,:]);
plt.xlabel("Subcarrier index");
plt.ylabel(r"|$h_\text{freq}$|");
plt.title("Channel frequency response");

In [ ]:
scene.preview()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def angle_delay_transform(h_freq):
    """
    - 先對 subcarriers (頻域) 做 IFFT -> 時延域 (Delay)
    - 再對 rx_antenna (空域) 做 FFT -> 角度域 (AOA)
    """
    # Step 1: IFFT over subcarriers (Convert to Delay domain)
    h_delay = np.fft.ifft(h_freq, axis=-1)

    # Step 2: FFT over rx_antenna (Convert to Angle domain)
    h_angle_delay = np.fft.fftshift(np.fft.fft(h_delay, axis=0), axes=0)

    return h_angle_delay


# **1. 取得「無干擾」場景的 CSI**
h_tx1_no_jamming = h_freq.numpy()[0, 0, :, 0, 0, 0, :]  # Tx1 in No Jamming scenario

# **2. 取得「有干擾」場景的 CSI**
h_tx1_jamming = h_freq.numpy()[0, 0, :, 0, 0, 0, :]  # Tx1 in Jamming scenario
h_tx2_jamming = h_freq.numpy()[0, 0, :, 1, 0, 0, :]  # Tx2 (Jammer) in Jamming scenario

# **3. 計算合併通道 (有干擾場景的 Rx 端接收通道)**
h_combined = h_tx1_jamming + h_tx2_jamming  # Rx 端實際接收的總信號

# **4. 進行 Angle-Delay 轉換**
H_no_jam_ad = angle_delay_transform(h_tx1_no_jamming)  # 無干擾 Tx1
H_jam_ad = angle_delay_transform(h_combined)           # 有干擾 (Tx1 + Tx2)

# 計算 dB 值
mag_no_jam = 20 * np.log10(np.abs(H_no_jam_ad) + 1e-9)
mag_jam = 20 * np.log10(np.abs(H_jam_ad) + 1e-9)

# 設定全局的色階範圍
vmin = min(mag_no_jam.min(), mag_jam.min())
vmax = max(mag_no_jam.max(), mag_jam.max())

plt.figure(figsize=(12,5))

# 無干擾場景
plt.subplot(1,2,1)
plt.imshow(mag_no_jam, aspect='auto', cmap='jet', vmin=vmin, vmax=vmax)
plt.colorbar(label="Magnitude (dB)")
plt.xlabel("Delay Index")
plt.ylabel("Angle Index")
plt.title("AOA vs. Delay (No Jamming)")

# 有干擾場景
plt.subplot(1,2,2)
plt.imshow(mag_jam, aspect='auto', cmap='jet', vmin=vmin, vmax=vmax)
plt.colorbar(label="Magnitude (dB)")
plt.xlabel("Delay Index")
plt.ylabel("Angle Index")
plt.title("AOA vs. Delay (Jamming)")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # 用於 3D 繪圖

def angle_delay_transform(h_freq):
    """
    - 先對 subcarriers (頻域) 做 IFFT -> 時延域 (Delay)
    - 再對 rx_antenna (空域) 做 FFT -> 角度域 (AOA)
    """
    # Step 1: IFFT over subcarriers (Convert to Delay domain)
    h_delay = np.fft.ifft(h_freq, axis=-1)
    # Step 2: FFT over rx_antenna (Convert to Angle domain)
    h_angle_delay = np.fft.fftshift(np.fft.fft(h_delay, axis=0), axes=0)
    return h_angle_delay

# **1. 取得「無干擾」場景的 CSI**
h_tx1_no_jamming = h_freq.numpy()[0, 0, :, 0, 0, 0, :]  # Tx1 in No Jamming scenario

# **2. 取得「有干擾」場景的 CSI**
h_tx1_jamming = h_freq.numpy()[0, 0, :, 0, 0, 0, :]  # Tx1 in Jamming scenario
h_tx2_jamming = h_freq.numpy()[0, 0, :, 1, 0, 0, :]  # Tx2 (Jammer) in Jamming scenario

# **3. 計算合併通道 (有干擾場景的 Rx 端接收通道)**
h_combined = h_tx1_jamming + h_tx2_jamming  # Rx 端實際接收的總信號

# **4. 進行 Angle-Delay 轉換**
H_no_jam_ad = angle_delay_transform(h_tx1_no_jamming)  # 無干擾 Tx1
H_jam_ad = angle_delay_transform(h_combined)           # 有干擾 (Tx1 + Tx2)

# 計算 dB 值
mag_no_jam = 20 * np.log10(np.abs(H_no_jam_ad) + 1e-9)
mag_jam    = 20 * np.log10(np.abs(H_jam_ad) + 1e-9)

# 設定全局的色階範圍
vmin = min(mag_no_jam.min(), mag_jam.min())
vmax = max(mag_no_jam.max(), mag_jam.max())

# 建立 delay 與 angle 的索引網格
# 假設 mag 的形狀為 (角度數, 延遲數)
angle_idx = np.arange(mag_no_jam.shape[0])
delay_idx = np.arange(mag_no_jam.shape[1])
Delay, Angle = np.meshgrid(delay_idx, angle_idx)

# 繪製 3D 圖
fig = plt.figure(figsize=(12,6))

# 3D 圖 - 無干擾場景
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf1 = ax1.plot_surface(Delay, Angle, mag_no_jam, cmap='jet', vmin=vmin, vmax=vmax)
fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=10, label="Magnitude (dB)")
ax1.set_xlabel("Delay Index")
ax1.set_ylabel("Angle Index")
ax1.set_zlabel("Magnitude (dB)")
ax1.set_title("AOA vs. Delay (No Jamming)")

# 3D 圖 - 有干擾場景
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
surf2 = ax2.plot_surface(Delay, Angle, mag_jam, cmap='jet', vmin=vmin, vmax=vmax)
fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=10, label="Magnitude (dB)")
ax2.set_xlabel("Delay Index")
ax2.set_ylabel("Angle Index")
ax2.set_zlabel("Magnitude (dB)")
ax2.set_title("AOA vs. Delay (Jamming)")

plt.tight_layout()
plt.show()


In [ ]:
# 計算 dB 值
mag_no_jam = 20 * np.log10(np.abs(H_no_jam_ad) + 1e-9)
mag_jam = 20 * np.log10(np.abs(H_jam_ad) + 1e-9)

# 計算差異：無干擾 - 干擾，正值表示無干擾時強度較高
diff_mag = mag_no_jam - mag_jam

plt.figure(figsize=(6,5))
plt.imshow(diff_mag, aspect='auto', cmap='bwr')
plt.colorbar(label="Magnitude Difference (dB)")
plt.xlabel("Delay Index")
plt.ylabel("Angle Index")
plt.title("Difference (No Jamming - Jamming)")

plt.tight_layout()
plt.show()


In [ ]:
ratio = (np.abs(H_no_jam_ad) + 1e-9) / (np.abs(H_jam_ad) + 1e-9)
ratio_db = 20 * np.log10(ratio)
plt.figure(figsize=(6,5))
plt.imshow(ratio_db, aspect='auto', cmap='bwr')
plt.colorbar(label="Ratio (dB)")
plt.xlabel("Delay Index")
plt.ylabel("Angle Index")
plt.title("Ratio (No Jamming / Jamming)")
plt.tight_layout()
plt.show()
